In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
vocab_size = 10
embed_dim = 4
embedding = nn.Embedding(vocab_size, embed_dim)

In [3]:
print(f"Embedding weight shape: {embedding.weight.shape}")

Embedding weight shape: torch.Size([10, 4])


In [4]:
embedding.weight

Parameter containing:
tensor([[ 1.7212,  0.0242,  0.2889,  0.5996],
        [ 0.2158, -1.6612, -1.3036, -0.8413],
        [ 2.4007,  1.8456, -1.1984,  0.1537],
        [-0.9187,  0.9768, -1.0372, -0.8833],
        [-0.6914, -0.7134, -0.1375,  0.4281],
        [-0.1266, -0.2050, -0.2823, -1.7913],
        [-1.1285, -0.3724,  1.5466,  0.3033],
        [-1.3213,  2.0505, -0.7692,  0.0617],
        [-1.3855, -1.0544,  0.8628, -0.1679],
        [ 0.9752, -0.3631, -1.1037,  1.1425]], requires_grad=True)

In [5]:
token_ids = torch.tensor([0, 3, 7])
vectors = embedding(token_ids)

In [6]:
vectors

tensor([[ 1.7212,  0.0242,  0.2889,  0.5996],
        [-0.9187,  0.9768, -1.0372, -0.8833],
        [-1.3213,  2.0505, -0.7692,  0.0617]], grad_fn=<EmbeddingBackward0>)

In [7]:
print(f"Token IDs: {token_ids}")
print(f"Vectors shape: {vectors.shape}")  # [3, 4]

Token IDs: tensor([0, 3, 7])
Vectors shape: torch.Size([3, 4])


In [8]:
# Semantic similarity via cosine similarity
torch.manual_seed(42)
embed_dim = 4
sim_embed = nn.Embedding(100, embed_dim)

In [9]:
# Simulate: "king"=12, "queen"=13, "apple"=50
king = sim_embed(torch.tensor(12))
queen = sim_embed(torch.tensor(13))
apple = sim_embed(torch.tensor(50))

In [10]:
cos_kq = F.cosine_similarity(king.unsqueeze(0), queen.unsqueeze(0)).item()
cos_ka = F.cosine_similarity(king.unsqueeze(0), apple.unsqueeze(0)).item()
print(f"\ncos(king, queen) = {cos_kq:.4f}")
print(f"cos(king, apple) = {cos_ka:.4f}")
print(f"(Random init — no real semantic meaning yet. Training creates it.)")


cos(king, queen) = -0.3744
cos(king, apple) = -0.7954
(Random init — no real semantic meaning yet. Training creates it.)


In [11]:
# --- Batched embedding lookup (how models actually use it) ---
batch_token_ids = torch.tensor([[1, 5, 3], [2, 7, 9]])  # [batch=2, seq=3]
batch_vectors = embedding(batch_token_ids)
print(f"\nBatch lookup [batch=2, sequence=3]: {batch_token_ids.shape} → {batch_vectors.shape}")


Batch lookup [batch=2, sequence=3]: torch.Size([2, 3]) → torch.Size([2, 3, 4])


In [12]:
print(f"weight: \n{embedding.weight}")

weight: 
Parameter containing:
tensor([[ 1.7212,  0.0242,  0.2889,  0.5996],
        [ 0.2158, -1.6612, -1.3036, -0.8413],
        [ 2.4007,  1.8456, -1.1984,  0.1537],
        [-0.9187,  0.9768, -1.0372, -0.8833],
        [-0.6914, -0.7134, -0.1375,  0.4281],
        [-0.1266, -0.2050, -0.2823, -1.7913],
        [-1.1285, -0.3724,  1.5466,  0.3033],
        [-1.3213,  2.0505, -0.7692,  0.0617],
        [-1.3855, -1.0544,  0.8628, -0.1679],
        [ 0.9752, -0.3631, -1.1037,  1.1425]], requires_grad=True)


In [13]:
print(f"batch_token_ids: \n{batch_token_ids}")

batch_token_ids: 
tensor([[1, 5, 3],
        [2, 7, 9]])


In [14]:
print(f"batch_vectors: \n{batch_vectors}")

batch_vectors: 
tensor([[[ 0.2158, -1.6612, -1.3036, -0.8413],
         [-0.1266, -0.2050, -0.2823, -1.7913],
         [-0.9187,  0.9768, -1.0372, -0.8833]],

        [[ 2.4007,  1.8456, -1.1984,  0.1537],
         [-1.3213,  2.0505, -0.7692,  0.0617],
         [ 0.9752, -0.3631, -1.1037,  1.1425]]], grad_fn=<EmbeddingBackward0>)


In [15]:
embed_dim

4

In [16]:
# --- Position embeddings (add word + position info) ---
seq_len = 3
word_embed = nn.Embedding(vocab_size, embed_dim)
pos_embed = nn.Embedding(seq_len, embed_dim)  # one vector per position 
# still random here, pos_embed can be: 1. learned 2. fixed Absolute PE 3. RoPE

positions = torch.arange(seq_len)               # [0, 1, 2]
word_vecs = word_embed(token_ids)               # [3, 4] — semantic meaning
pos_vecs = pos_embed(positions)                 # [3, 4] — positional info
combined = word_vecs + pos_vecs                 # [3, 4] — meaning + position
print(f"\nCombined (word + position): {combined.shape}")


Combined (word + position): torch.Size([3, 4])
